In [2]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import shutil
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

# Настройка GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

BASE_DIR = "data"
CLASSES = ['0', '1', '3', '8']
NUM_CLASSES = len(CLASSES)

def prepare_data_splits():
    """Разделяет исходные данные на указанные пропорции для каждого сценария."""
    splits_config = {
        '100_15_20': (100, 15, 20),
        '250_30_20': (250, 30, 20),
        '400_60_20': (400, 60, 20),
        '700_100_20': (700, 80, 20)
    }
    
    src_train = os.path.join(BASE_DIR, 'train')
    
    for split_name, (n_train, n_val, n_test) in splits_config.items():
        split_base = os.path.join(BASE_DIR, f'split_{split_name}')
        for part in ['train', 'val', 'test']:
            for cls in CLASSES:
                os.makedirs(os.path.join(split_base, part, cls), exist_ok=True)
                
        for cls in CLASSES:
            cls_dir = os.path.join(src_train, cls)
            files = [f for f in os.listdir(cls_dir) if f.endswith('.png')]
            random.shuffle(files)
            
            selected = files[:n_train + n_val + n_test]
            partitions = {
                'train': selected[:n_train],
                'val': selected[n_train:n_train + n_val],
                'test': selected[n_train + n_val:]
            }
            
            for part, imgs in partitions.items():
                for img in imgs:
                    src = os.path.join(cls_dir, img)
                    dst = os.path.join(split_base, part, cls, img)
                    if not os.path.exists(dst):
                        shutil.copy(src, dst)

def get_model_config(model_name):
    """Возвращает архитектуру и требуемый размер входа для модели."""
    sizes = {
        'Xception': (224, 224),
        'ResNet50V2': (224, 224),
        'InceptionResNetV2': (139, 139),
        'DenseNet201': (224, 224),
        'NASNetLarge': (331, 331)
    }
    base_model = getattr(keras.applications, model_name)(
        weights='imagenet',
        include_top=False,
        input_shape=(sizes[model_name][0], sizes[model_name][1], 3)
    )
    base_model.trainable = False
    return base_model, sizes[model_name]

def build_model(model_name):
    """Создает модель переноса обучения с замороженной базой и кастомным классификатором."""
    base_model, input_size = get_model_config(model_name)
    
    inputs = keras.Input(shape=(input_size[0], input_size[1], 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model, input_size

def run_training(model_name, split_name, epochs=10, batch_size=16):
    """Обучает модель, оценивает на тесте и возвращает метрики."""
    split_dir = os.path.join(BASE_DIR, f'split_{split_name}')
    model, img_size = build_model(model_name)
    
    ds_kwargs = dict(image_size=img_size, batch_size=batch_size, label_mode='categorical')
    train_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(split_dir, 'train'), **ds_kwargs, shuffle=True)
    val_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(split_dir, 'val'), **ds_kwargs, shuffle=False)
    test_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(split_dir, 'test'), **ds_kwargs, shuffle=False)
    
    # Оптимизация пайплайна данных
    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(autotune)
    val_ds = val_ds.cache().prefetch(autotune)
    test_ds = test_ds.cache().prefetch(autotune)
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(f'weights_{model_name}_{split_name}.keras', monitor='val_accuracy', save_best_only=True, mode='max')
    ]
    
    model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=1)
    
    # Оценка на тестовой выборке
    y_true_onehot = np.concatenate([y for x, y in test_ds], axis=0)
    y_pred_probs = model.predict(test_ds)
    y_true = np.argmax(y_true_onehot, axis=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    keras.backend.clear_session()
    
    return {
        'Network': model_name,
        'Split': split_name,
        'Accuracy': report['accuracy'],
        'Precision': report['macro avg']['precision'],
        'Recall': report['macro avg']['recall'],
        'F1': report['macro avg']['f1-score'],
        'Confusion_Matrix': cm.tolist()
    }

def main():
    prepare_data_splits()
    
    models = ['Xception', 'ResNet50V2', 'InceptionResNetV2', 'DenseNet201', 'NASNetLarge']
    splits = ['100_15_20', '250_30_20', '400_60_20', '700_100_20']
    
    results = []
    for split in splits:
        for model in models:
            print(f"[INFO] Training {model} on {split}")
            results.append(run_training(model, split))
            
    df = pd.DataFrame(results)
    df.to_excel('lab12_results.xlsx', index=False)
    
    # Вывод топ-3 моделей по F1-мере
    top3 = df.nlargest(3, 'F1')[['Network', 'Split', 'F1']]
    print("\n[INFO] Top 3 models by F1-Score:")
    print(top3.to_string(index=False))
    print(f"\n[INFO] Results saved to lab12_results.xlsx")

if __name__ == '__main__':
    main()

[INFO] Training Xception on 100_15_20
Found 1513 files belonging to 4 classes.
Found 268 files belonging to 4 classes.
Found 362 files belonging to 4 classes.
Epoch 1/10
95/95 [==============================] - 11s 52ms/step - loss: 21.7603 - accuracy: 0.2750 - val_loss: 1.3761 - val_accuracy: 0.2910
Epoch 2/10
95/95 [==============================] - 4s 40ms/step - loss: 1.3772 - accuracy: 0.2624 - val_loss: 1.3737 - val_accuracy: 0.2910
Epoch 3/10
95/95 [==============================] - 4s 39ms/step - loss: 1.3834 - accuracy: 0.2498 - val_loss: 1.3836 - val_accuracy: 0.2687
Epoch 4/10
95/95 [==============================] - 4s 41ms/step - loss: 1.3807 - accuracy: 0.2631 - val_loss: 1.3701 - val_accuracy: 0.2612
Epoch 5/10
95/95 [==============================] - 4s 39ms/step - loss: 1.3835 - accuracy: 0.2611 - val_loss: 1.3762 - val_accuracy: 0.2761
Epoch 6/10
95/95 [==============================] - 4s 40ms/step - loss: 1.3837 - accuracy: 0.2644 - val_loss: 1.3840 - val_accuracy: 